In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [ ]:
df=pd.read_csv('data/gandhinagar_property_apartments_cleaned.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1304 entries, 0 to 1303
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Name                      1304 non-null   str    
 1   Location                  1304 non-null   str    
 2   Price (INR in Lakhs)      1304 non-null   float64
 3   Price_per_sqft            1304 non-null   float64
 4   Area_sqft                 1304 non-null   float64
 5   Description               1203 non-null   str    
 6   Property_URL              1304 non-null   str    
 7   bedrooms                  1302 non-null   float64
 8   bathrooms                 1304 non-null   float64
 9   balconies                 1304 non-null   float64
 10  current_floor             1304 non-null   float64
 11  total_floors              1304 non-null   float64
 12  furnishing_status         1304 non-null   str    
 13  Mapped_Area               1304 non-null   str    
 14  facing             

In [46]:
pd.set_option('display.max_columns', None)
# for shoing full values in column
pd.set_option('display.max_colwidth', None)

In [ ]:
df.loc[16,'is_carpet_area']=True
df.loc[46,'Area_sqft']=1170
df.loc[46,'is_carpet_area']=True
df.loc[60,'is_carpet_area']=True
df.loc[103,'is_super_built_up_area']=True
df.loc[154,'Area_sqft']=1122.56
df.loc[154,'is_carpet_area']=True
df.loc[189,'is_carpet_area']=True
df.loc[205,'is_super_built_up_area']=True
df.loc[210,'Area_sqft']=1071
df.loc[210,'is_carpet_area']=True
df.loc[241,'is_carpet_area']=True
df.loc[264,'is_carpet_area']=True
df.loc[400,'is_super_built_up_area']=True
df.loc[469,'is_carpet_area']=True
df.loc[519,'Area_sqft']=2025
df.loc[519,'is_super_built_up_area']=True
df.loc[536,'is_carpet_area']=True
df.loc[571,'is_carpet_area']=True
df.loc[574,'is_carpet_area']=True
df.loc[604,'is_carpet_area']=True
df.loc[615,'is_carpet_area']=True
df.loc[617,'Area_sqft']=1242
df.loc[617,'is_super_built_up_area']=True
df.loc[681,'is_carpet_area']=True
df.loc[703,'is_super_built_up_area']=True
df.loc[774,'is_super_built_up_area']=True
df.loc[792,'is_super_built_up_area']=True
df.loc[794,'is_built_up_area']=True
df.loc[795,'is_built_up_area']=True
df.loc[803,'is_carpet_area']=True
df.loc[821,'is_carpet_area']=True
df.loc[829,'is_carpet_area']=True
df.loc[890,'is_built_up_area']=True
df.loc[949,'Area_sqft']=765
df.loc[949,'is_built_up_area']=True
df.loc[954,'is_super_built_up_area']=True
df.loc[966,'is_built_up_area']=True
df.loc[988,'is_carpet_area']=True
df.loc[1021,'is_carpet_area']=True
df.loc[1035,'Area_sqft']=2043
df.loc[1035,'is_super_built_up_area']=True
df.loc[1166,'is_carpet_area']=True
df.loc[1196,'is_carpet_area']=True
df.loc[1226,'is_carpet_area']=True
df.loc[1297,'is_carpet_area']=True
df.loc[1300,'is_super_built_up_area']=True

In [ ]:
df[(df['is_built_up_area']==False) & (df['is_super_built_up_area']==False) & (df['is_carpet_area']==False)][['Name','Bedrooms', 'Property_URL','Area_sqft', 'is_built_up_area', 'is_super_built_up_area', 'is_carpet_area']]

,Name,Bedrooms,Property_URL,Area_sqft,is_built_up_area,is_super_built_up_area,is_carpet_area


In [57]:
# Create Area_type column
df['Area_type'] = np.select(
    [
        df['is_super_built_up_area'] == 1,
        df['is_built_up_area'] == 1,
        df['is_carpet_area'] == 1
    ],
    [
        'Super Built-up',
        'Built-up',
        'Carpet'
    ],
    default='Unknown'
)

# Drop old columns
df.drop(
    columns=[
        'is_super_built_up_area',
        'is_built_up_area',
        'is_carpet_area'
    ],
    inplace=True
)

In [59]:
df['Status'] = df['is_ready_to_move'].map({
    1: 'Ready_to_Move',
    0: 'Under_Construction'
})

df.drop(columns=['is_ready_to_move'], inplace=True)

In [61]:
df['Area_type'] = df['Area_type'].astype('category')
df['Status'] = df['Status'].astype('category')

In [62]:
print(df['Area_type'].value_counts())
print(df['Status'].value_counts())

Area_type
Super Built-up    649
Carpet            527
Built-up          128
Name: count, dtype: int64
Status
Ready_to_Move         1061
Under_Construction     243
Name: count, dtype: int64


In [63]:
df.drop(columns=['Name', 'Location', 'Price_per_sqft', 'Description',
                 'Property_URL','bedrooms','property_type','luxury_score',
                 'location_advantage_score','log_price','pps'], inplace=True)

In [64]:
df.head()

,Price (INR in Lakhs),Area_sqft,bathrooms,balconies,current_floor,total_floors,furnishing_status,Mapped_Area,facing,property_age_bucket,Bedrooms,Area_type,Status
0,126.00,2916.0,3.0,1.0,4.0,8.0,Unknown,Sargasan,Unknown,New,3.0,Super Built-up,Ready_to_Move
1,51.99,1755.0,3.0,2.0,3.0,7.0,Unknown,Pethapur,Unknown,New,3.0,Super Built-up,Under_Construction
2,97.00,1908.0,3.0,2.0,4.0,8.0,Unknown,Raysan,Unknown,New,3.0,Super Built-up,Ready_to_Move
3,125.00,2205.0,2.0,2.0,9.0,13.0,Unknown,Raysan,Unknown,New,3.0,Carpet,Ready_to_Move
4,79.00,1755.0,3.0,1.0,8.0,13.0,unfurnished,Randesan,Unknown,5-10 years,3.0,Carpet,Ready_to_Move


In [65]:
df.rename(columns={'Area_sqft': 'Area',
                   'Price (INR in Lakhs)':'Price',
                   'bathrooms':'Bathrooms',
                   'balconies':'Balconies',
                   'current_floor':'Current_Floor',
                   'total_floors':'Total_Floors',
                   'furnishing_status':'Furnishing_Status',
                   'facing':'Facing',
                    'Area_type':'Area_Type',
                    'property_age_bucket':'Property_Age',
                    'Status':'Property_Status',   
                   },
          inplace=True)

In [66]:
df.head()

,Price,Area,Bathrooms,Balconies,Current_Floor,Total_Floors,Furnishing_Status,Mapped_Area,Facing,Property_Age,Bedrooms,Area_Type,Property_Status
0,126.00,2916.0,3.0,1.0,4.0,8.0,Unknown,Sargasan,Unknown,New,3.0,Super Built-up,Ready_to_Move
1,51.99,1755.0,3.0,2.0,3.0,7.0,Unknown,Pethapur,Unknown,New,3.0,Super Built-up,Under_Construction
2,97.00,1908.0,3.0,2.0,4.0,8.0,Unknown,Raysan,Unknown,New,3.0,Super Built-up,Ready_to_Move
3,125.00,2205.0,2.0,2.0,9.0,13.0,Unknown,Raysan,Unknown,New,3.0,Carpet,Ready_to_Move
4,79.00,1755.0,3.0,1.0,8.0,13.0,unfurnished,Randesan,Unknown,5-10 years,3.0,Carpet,Ready_to_Move


In [ ]:
df.to_csv("gandhinagar_property_apartments_final.csv",index=False)